# AgriLink — Person 2: FP-Growth Recommendation System

**Purpose:** Discover frequently purchased crop combinations from order baskets and use the resulting association rules to generate customer-specific recommendations.

This notebook covers **only the FP-Growth / Association Recommendation section**:
1. Create/load purchase data
2. Create transaction/basket matrix
3. Run FP-Growth
4. Generate association rules
5. Analyze support, confidence and lift
6. Apply rules to a customer's basket
7. Produce the common integration output:
   `customer_id, listing_id, crop_id, crop_name, score, recommendation_type, reason`

> The synthetic data below is for development/testing before sufficient real transaction data is available. Replace the data-loading section with Supabase data when the project database is ready.

In [ ]:
# COMMAND: Install required libraries
!pip install -q pandas numpy mlxtend supabase

In [ ]:
# COMMAND: Import libraries
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules
import random
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

## 1. Crop Catalogue

The current catalogue contains the 15 agricultural crops used for testing.

In [ ]:
# COMMAND: Define crop catalogue

crops = [
    "Apple",
    "Banana",
    "Beans",
    "Brinjal",
    "Cabbage",
    "Carrot",
    "Coriander",
    "Cucumber",
    "Green Chilli",
    "Onion",
    "Orange",
    "Papaya",
    "Potato",
    "Rice",
    "Tomato"
]

print("Number of crops:", len(crops))
print("Crops:")
print(", ".join(crops))

## 2. Create Synthetic Order Data

FP-Growth works on **transactions/baskets**. Each order represents one basket containing one or more crops.

The generated data intentionally contains repeated basket patterns so that meaningful product associations can be discovered.

In [ ]:
# COMMAND: Define realistic basket patterns

shopping_patterns = [
    ["Tomato", "Onion", "Potato", "Brinjal", "Beans", "Coriander"],
    ["Tomato", "Onion", "Potato", "Coriander"],
    ["Cucumber", "Carrot", "Coriander"],
    ["Tomato", "Onion", "Green Chilli", "Coriander"],
    ["Apple", "Banana", "Orange", "Papaya"],
    ["Rice", "Potato", "Onion", "Tomato"],
    ["Tomato", "Onion", "Potato", "Carrot", "Banana"],
    ["Tomato", "Onion", "Potato", "Cabbage", "Carrot", "Beans", "Coriander"]
]

for i, pattern in enumerate(shopping_patterns, start=1):
    print(f"Pattern {i}: {', '.join(pattern)}")

In [ ]:
# COMMAND: Generate synthetic customer orders

random.seed(42)
np.random.seed(42)

NUM_CUSTOMERS = 1000
MIN_ORDERS_PER_CUSTOMER = 3
MAX_ORDERS_PER_CUSTOMER = 10

records = []
order_counter = 1000

for customer_id in range(1, NUM_CUSTOMERS + 1):

    num_orders = random.randint(
        MIN_ORDERS_PER_CUSTOMER,
        MAX_ORDERS_PER_CUSTOMER
    )

    for _ in range(num_orders):

        order_counter += 1

        # Select a basket pattern
        pattern = random.choice(shopping_patterns)

        # Select a subset of products from that pattern
        min_products = max(2, len(pattern) // 2)
        max_products = len(pattern)
        num_products = random.randint(min_products, max_products)

        selected_products = random.sample(
            pattern,
            num_products
        )

        for product in selected_products:
            records.append({
                "customer_id": customer_id,
                "order_id": order_counter,
                "crop_name": product
            })

purchase_data = pd.DataFrame(records)

print("Purchase dataset created.")
print("Rows:", len(purchase_data))
print("Customers:", purchase_data["customer_id"].nunique())
print("Orders:", purchase_data["order_id"].nunique())
print("Crops:", purchase_data["crop_name"].nunique())

display(purchase_data.head(20))

In [ ]:
# COMMAND: Basic data validation

print("Missing values:")
display(purchase_data.isnull().sum().to_frame("missing_values"))

print("\nData types:")
display(purchase_data.dtypes.to_frame("dtype"))

print("\nSample orders:")
display(
    purchase_data
    .groupby("order_id")["crop_name"]
    .apply(list)
    .reset_index()
    .head(10)
)

## 3. Create Transaction / Basket Matrix

Each row is an order.

Each crop becomes a column.

- `1` = crop is present in that order
- `0` = crop is not present in that order

This is the input format required by `mlxtend` FP-Growth.

In [ ]:
# COMMAND: Create basket matrix

basket = (
    purchase_data
    .assign(value=1)
    .pivot_table(
        index="order_id",
        columns="crop_name",
        values="value",
        aggfunc="max",
        fill_value=0
    )
)

# Ensure every catalogue crop exists as a column
basket = basket.reindex(columns=crops, fill_value=0)

# FP-Growth expects boolean values
basket = basket.astype(bool)

print("Basket shape:", basket.shape)
display(basket.head(10))

In [ ]:
# COMMAND: Verify basket matrix

print("Unique values in basket:", np.unique(basket.astype(int).values))
print("Number of transactions:", len(basket))
print("Number of product columns:", len(basket.columns))

assert set(np.unique(basket.astype(int).values)).issubset({0, 1})

print("Basket validation passed.")

## 4. Run FP-Growth

`min_support` determines how frequently an itemset must occur in the transaction dataset to be considered frequent.

For this project, the starting value is **0.05 (5%)**, matching the project specification.

Support is:

> Number of transactions containing the itemset / Total number of transactions

In [ ]:
# COMMAND: Run FP-Growth

MIN_SUPPORT = 0.05

frequent_itemsets = fpgrowth(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

frequent_itemsets["itemset_size"] = (
    frequent_itemsets["itemsets"].apply(len)
)

frequent_itemsets = frequent_itemsets.sort_values(
    by=["itemset_size", "support"],
    ascending=[True, False]
).reset_index(drop=True)

print("Minimum support:", MIN_SUPPORT)
print("Number of frequent itemsets:", len(frequent_itemsets))

display(frequent_itemsets.head(30))

In [ ]:
# COMMAND: View frequent itemsets by size

for size in sorted(frequent_itemsets["itemset_size"].unique()):
    count = (frequent_itemsets["itemset_size"] == size).sum()
    print(f"Itemsets containing {size} product(s): {count}")

print("\nTop frequent itemsets:")
display(
    frequent_itemsets[
        ["itemsets", "support", "itemset_size"]
    ].head(30)
)

## 5. Generate Association Rules

Association rules turn frequent itemsets into relationships such as:

`Tomato + Onion → Coriander`

The project specification uses:
- confidence as the rule-generation metric
- minimum confidence = 0.30

Important metrics:
- **Support:** how common the complete rule combination is
- **Confidence:** how often the consequent appears when the antecedent appears
- **Lift:** how much stronger the relationship is compared with the consequent occurring independently

In [ ]:
# COMMAND: Generate association rules

MIN_CONFIDENCE = 0.30

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

print("Minimum confidence:", MIN_CONFIDENCE)
print("Number of generated rules:", len(rules))

display(
    rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(30)
)

In [ ]:
# COMMAND: Keep and format required rule fields

rules = rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].copy()

def format_itemset(itemset):
    return " + ".join(sorted(list(itemset)))

rules["antecedents"] = rules["antecedents"].apply(format_itemset)
rules["consequents"] = rules["consequents"].apply(format_itemset)

rules = rules.sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
).reset_index(drop=True)

display(rules.head(30))

In [ ]:
# COMMAND: Keep stronger positive-association rules

strong_rules = rules[
    rules["lift"] > 1
].copy()

strong_rules = strong_rules.sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
).reset_index(drop=True)

print("Rules with lift > 1:", len(strong_rules))
display(strong_rules.head(30))

## 6. Customer-Specific Association Recommendations

FP-Growth discovers general relationships between products.

To personalize those relationships for a customer:

1. Find products in the customer's previous orders.
2. Find association rules whose antecedent is contained in that customer's purchased products.
3. Take the consequent as a candidate recommendation.
4. Remove products the customer already purchased.
5. Rank candidates using a project-defined recommendation score.

For this MVP, the ranking score is:

`score = confidence × lift`

This score is used only for ranking; it is **not an intrinsic FP-Growth output**.

In [ ]:
# COMMAND: Get products purchased by a customer

def get_customer_products(customer_id):
    return set(
        purchase_data.loc[
            purchase_data["customer_id"] == customer_id,
            "crop_name"
        ]
    )

# Example
customer_id = 1

customer_products = get_customer_products(customer_id)

print("Customer:", customer_id)
print("Purchased products:")
print(", ".join(sorted(customer_products)))

In [ ]:
# COMMAND: Generate FP-Growth recommendations for a customer

def get_fp_growth_recommendations(customer_id, top_n=10):

    customer_products = get_customer_products(customer_id)

    if not customer_products:
        return pd.DataFrame()

    recommendations = []

    for _, rule in rules.iterrows():

        antecedent_products = set(
            rule["antecedents"].split(" + ")
        )

        consequent_products = set(
            rule["consequents"].split(" + ")
        )

        # Rule applies if customer has every
        # product in its antecedent
        if antecedent_products.issubset(customer_products):

            for product in consequent_products:

                # Do not recommend products already purchased
                if product not in customer_products:

                    recommendations.append({
                        "customer_id": customer_id,
                        "crop_name": product,
                        "support": rule["support"],
                        "confidence": rule["confidence"],
                        "lift": rule["lift"],
                        "trigger": rule["antecedents"]
                    })

    if not recommendations:
        return pd.DataFrame()

    recommendations = pd.DataFrame(recommendations)

    # One product can be recommended by several rules.
    # Keep the strongest rule for each product.
    recommendations = (
        recommendations
        .sort_values(
            by=["lift", "confidence", "support"],
            ascending=False
        )
        .drop_duplicates(
            subset=["customer_id", "crop_name"]
        )
    )

    # Project-defined ranking score
    recommendations["score"] = (
        recommendations["confidence"]
        * recommendations["lift"]
    )

    recommendations = (
        recommendations
        .sort_values(
            by="score",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    return recommendations

In [ ]:
# COMMAND: Test customer-specific recommendations

customer_id = 1

customer_products = get_customer_products(customer_id)

print("Customer:", customer_id)
print("\nCustomer basket/history:")
print(", ".join(sorted(customer_products)))

recommendations = get_fp_growth_recommendations(
    customer_id=customer_id,
    top_n=10
)

print("\nRecommendations:")

if recommendations.empty:
    print("No association-based recommendations found.")
else:
    display(recommendations)

## 7. Create Crop and Listing Mapping for Integration Testing

The final project requires:

`customer_id, listing_id, crop_id, crop_name, score, recommendation_type, reason`

The synthetic dataset does not contain real `crop_id` or `listing_id`, so temporary IDs are created here only for testing.

When the real database is connected, these values should come from the project's `crop` and `seller_listing` tables.

In [ ]:
# COMMAND: Create temporary crop and listing mappings

crop_mapping = pd.DataFrame({
    "crop_id": range(1, len(crops) + 1),
    "crop_name": crops
})

listing_mapping = crop_mapping.copy()

listing_mapping["listing_id"] = (
    listing_mapping["crop_id"] + 100
)

listing_mapping = listing_mapping[
    ["listing_id", "crop_id", "crop_name"]
]

display(listing_mapping)

In [ ]:
# COMMAND: Generate final common integration output

def generate_final_fp_growth_output(customer_id, top_n=10):

    recommendations = get_fp_growth_recommendations(
        customer_id=customer_id,
        top_n=top_n
    )

    output_columns = [
        "customer_id",
        "listing_id",
        "crop_id",
        "crop_name",
        "score",
        "recommendation_type",
        "reason"
    ]

    if recommendations.empty:
        return pd.DataFrame(columns=output_columns)

    recommendations = recommendations.merge(
        listing_mapping,
        on="crop_name",
        how="left"
    )

    recommendations["recommendation_type"] = "ASSOCIATION"

    recommendations["reason"] = (
        "Frequently purchased with "
        + recommendations["trigger"]
    )

    final_output = recommendations[
        output_columns
    ].copy()

    return final_output

In [ ]:
# COMMAND: Test final integration output

customer_id = 1

final_recommendations = generate_final_fp_growth_output(
    customer_id=customer_id,
    top_n=5
)

display(final_recommendations)

## 8. Test Multiple Customers

In [ ]:
# COMMAND: Generate recommendations for multiple customers

test_customers = [1, 2, 3, 4, 5]

all_test_recommendations = []

for customer_id in test_customers:

    result = generate_final_fp_growth_output(
        customer_id=customer_id,
        top_n=5
    )

    if not result.empty:
        all_test_recommendations.append(result)

if all_test_recommendations:
    all_test_recommendations = pd.concat(
        all_test_recommendations,
        ignore_index=True
    )
else:
    all_test_recommendations = pd.DataFrame(
        columns=[
            "customer_id",
            "listing_id",
            "crop_id",
            "crop_name",
            "score",
            "recommendation_type",
            "reason"
        ]
    )

display(all_test_recommendations)

In [ ]:
# COMMAND: Verify final output format

expected_columns = [
    "customer_id",
    "listing_id",
    "crop_id",
    "crop_name",
    "score",
    "recommendation_type",
    "reason"
]

assert list(all_test_recommendations.columns) == expected_columns

if not all_test_recommendations.empty:
    assert (
        all_test_recommendations["recommendation_type"]
        .eq("ASSOCIATION")
        .all()
    )

print("Final output format validation passed.")
print("Columns:", list(all_test_recommendations.columns))

## 9. Save FP-Growth Outputs

These files can later be given to the integration person/team.

- `fp_growth_purchase_data.csv` — synthetic purchase data
- `fp_growth_basket.csv` — transaction matrix
- `fp_growth_frequent_itemsets.csv` — FP-Growth output
- `fp_growth_association_rules.csv` — association rules
- `fp_growth_recommendations.csv` — final recommendation output

In [ ]:
# COMMAND: Save all important outputs

purchase_data.to_csv(
    "fp_growth_purchase_data.csv",
    index=False
)

basket.astype(int).to_csv(
    "fp_growth_basket.csv"
)

frequent_itemsets.to_csv(
    "fp_growth_frequent_itemsets.csv",
    index=False
)

rules.to_csv(
    "fp_growth_association_rules.csv",
    index=False
)

all_test_recommendations.to_csv(
    "fp_growth_recommendations.csv",
    index=False
)

print("All FP-Growth output files saved successfully.")

## 10. Final Summary

### Input
Historical customer orders containing multiple crops per order.

### Processing
`Orders → Basket Matrix → FP-Growth → Frequent Itemsets → Association Rules`

### Recommendation
`Customer's purchased products → Matching rules → Unpurchased consequent products → Ranked recommendations`

### Final output
`customer_id | listing_id | crop_id | crop_name | score | recommendation_type | reason`

For this section:

- FP-Growth is used to discover frequent itemsets.
- Association rules are generated using confidence.
- Support, confidence and lift are retained for analysis.
- Customer-specific recommendations are generated by matching a customer's purchased products against rule antecedents.
- `ASSOCIATION` is used as the recommendation type.
- The ranking score is `confidence × lift` and is a project-defined ranking score.
- Synthetic transaction data is used for development/testing until real transaction data is available.